# Kronos-NSE — evaluation grid on Kaggle

> Research/education tool — scenario visualization, not investment advice.

Runs the A3 evaluation grid on a Kaggle T4/P100 (16 GB), which finishes in roughly 1–2 hours
against 4–6 on a 6 GB laptop — and without the GPU dropping off the bus.

## Before you run anything

In the notebook sidebar:

1. **Accelerator → GPU T4 x2** (or P100). Without this the grid will not finish.
2. **Internet → On.** Needed to clone the repo and download the model weights.
3. **Attach the corpus dataset** — see the next cell.

## Getting `data/` here

`data/` is gitignored, so it does not arrive with the clone. On your local machine, zip
`data/` (17.6 MB) and upload it as a **Kaggle Dataset** (Datasets → New Dataset), then
attach it to this notebook. It will appear under `/kaggle/input/<your-dataset-name>/`.

**Do not re-run `fetch_nse.py` here.** Canonical prices are split/bonus back-adjusted, and
back-adjustment rewrites history — a corpus fetched on a different date is a *different
corpus*. Nothing would error; the numbers would simply stop being comparable to every
measurement taken so far.

## How to run it

Use **Save Version -> Save & Run All (Commit)**, not the interactive editor. An
interactive session is capped at a shorter idle timeout and dies when the browser tab
does; a committed run executes headlessly to the 12-hour limit and keeps its output.
Set `LIMIT = 60` for a first pass, confirm the port check passes, then commit the full
grid.

Every cell raises on failure, so a committed run that reports success really did
finish the grid.

## A note on exact reproduction

The baseline numbers in the verification cell are pure NumPy and **must** match exactly —
they prove the corpus and harness survived the move. The Kronos ensembles may differ in
the last digits from a different GPU architecture, which is normal. Do not mix devices
*within* one grid; finish a run on the hardware it started on, or resume on the same kind.

In [ ]:
import glob
import os
import pathlib
import shutil
import subprocess
import sys

# ---- configure -------------------------------------------------------------
REPO_URL = "https://github.com/neopentane7/kronos-candlecast.git"
DATA_DIR = None  # e.g. "/kaggle/input/kronos-nse-corpus"; None = autodetect

# Private repo? Add a GitHub token as a Kaggle Secret named GITHUB_TOKEN
# (Add-ons -> Secrets). Leave this alone if the repo is public.
USE_TOKEN = True

BATCH_SIZE = 24  # 16 GB card; drop to 12 if you hit OOM
CHECKPOINT_EVERY = 5  # flush partials every N batches
SPLIT = "test"
LIMIT = None  # e.g. 60 for a quick smoke run; None = full 708-window grid

WORK = pathlib.Path("/kaggle/working")
REPO = WORK / "kronos-candlecast"
print("working dir:", WORK)

In [ ]:
# ---- 1. clone the project and the pinned upstream --------------------------
UPSTREAM_SHA = "67b630e67f6a18c9e9be918d9b4337c960db1e9a"
UPSTREAM_URL = "https://github.com/shiyu-coder/Kronos.git"


def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True).stdout.strip()


url = REPO_URL
if USE_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{tok}@")
        print("using GITHUB_TOKEN from Kaggle Secrets")
    except Exception as exc:  # noqa: BLE001 - fall back to an anonymous clone
        print(f"no usable token ({type(exc).__name__}); trying anonymous clone")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True)
# The clone URL is written verbatim into .git/config, so a token embedded in it
# outlives the cell that read it -- `git remote -v`, and anything that dumps the
# config into the notebook output, would echo the secret. Rewrite the remote to
# the clean URL now that the fetch is done (working rule 4: secrets stay in env).
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print("repo:", git("-C", str(REPO), "log", "--oneline", "-1"))

# Upstream is gitignored by design and reproduced at its pinned commit. The harness
# overlays its generation loop and an equivalence test asserts bit-identical output,
# so the SHA is not optional.
up = REPO / "phase-a" / "Kronos"
if not up.exists():
    subprocess.run(["git", "clone", "--quiet", UPSTREAM_URL, str(up)], check=True)
    subprocess.run(
        ["git", "-C", str(up), "checkout", "--quiet", "--detach", UPSTREAM_SHA], check=True
    )
print("upstream:", git("-C", str(up), "rev-parse", "HEAD")[:12])

In [ ]:
# ---- 2. dependencies -------------------------------------------------------
# Kaggle already ships torch with a working CUDA build, so we install only what is
# missing rather than re-resolving the lockfile (which would pull a ~2.5 GB torch
# wheel). The trade-off: exact package pinning is relaxed.
#
# Minimum versions are pinned rather than bare names. `pip install pandera` is a
# no-op when ANY pandera is already present -- pip reports "already satisfied" and
# does not upgrade -- and Kaggle's base image ships many of these. Without the
# bounds, an old pandera would survive and `import pandera.pandas` would fail
# halfway through the run.
REQUIREMENTS = [
    "pandera>=0.24",  # the pandera.pandas namespace
    "scoringrules>=0.9",  # crps_ensemble(estimator="fair")
    "numpy>=1.22",  # np.quantile(method="weibull")
    "duckdb",
    "exchange_calendars",
    "einops",
]
!pip install -q -U {" ".join(f'"{r}"' for r in REQUIREMENTS)} 2>&1 | tail -3

# Verify the APIs we actually call, not the version strings. A satisfied version
# constraint is not proof the function exists with the signature we use.
import numpy as np  # noqa: E402
import pandera.pandas as pa  # noqa: E402  - namespace only exists in pandera >= 0.24
import scoringrules as sr  # noqa: E402
import torch  # noqa: E402

checks = {
    "np.quantile(method='weibull')": lambda: np.quantile([1.0, 2, 3], 0.5, method="weibull"),
    "sr.crps_ensemble(estimator='fair')": lambda: sr.crps_ensemble(
        np.zeros(2), np.zeros((2, 8)), m_axis=-1, estimator="fair"
    ),
    "pa.DataFrameSchema": lambda: pa.DataFrameSchema(columns={}),
}
broken = []
for name, probe in checks.items():
    try:
        probe()
        print(f"  ok   {name}")
    except Exception as exc:  # noqa: BLE001 - report every failure, not just the first
        print(f"  FAIL {name}: {type(exc).__name__}: {exc}")
        broken.append(name)
if broken:
    raise SystemExit(f"unusable environment: {broken}. Restart the kernel and re-run.")

print(f"\ntorch {torch.__version__} | cuda {torch.version.cuda}")
print(f"cuda available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Set Accelerator -> GPU T4 x2 in the sidebar.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}, {props.total_memory / 2**30:.1f} GiB")

In [ ]:
# ---- 3. restore the corpus -------------------------------------------------
src = DATA_DIR
if src is None:
    hits = glob.glob("/kaggle/input/*/parquet") + glob.glob("/kaggle/input/*/data/parquet")
    if not hits:
        raise SystemExit(
            "No corpus found under /kaggle/input. Upload data/ as a Kaggle Dataset "
            "and attach it, or set DATA_DIR explicitly."
        )
    src = str(pathlib.Path(hits[0]).parent)
    print("autodetected corpus at", src)

dest = REPO / "data"
dest.mkdir(exist_ok=True)
# `if target.exists(): continue` was wrong: a session that died mid-copy leaves a
# partial data/parquet, and skipping it would hand the grid a silently truncated
# corpus. copytree(dirs_exist_ok=True) fills in what is missing instead.
for item in pathlib.Path(src).iterdir():
    target = dest / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# 59 tickers, one partition each. An exact count, not `> 0`: a truncated corpus is
# the failure mode that produces plausible-looking numbers that mean nothing, and
# the port check in the next cell only catches it because the baselines shift.
EXPECTED_PARTITIONS = 59
n_parts = len(list((dest / "parquet").glob("*/*.parquet")))
print(f"corpus restored: {n_parts} ticker partitions")
assert n_parts == EXPECTED_PARTITIONS, (
    f"expected {EXPECTED_PARTITIONS} parquet partitions, found {n_parts} -- the "
    "dataset is truncated or its layout differs from the local corpus"
)

In [ ]:
# ---- 4. VERIFY THE PORT ----------------------------------------------------
# Baselines are pure NumPy and deterministic from the grid and the seed, so these
# numbers must reproduce exactly on any machine. If they do, the corpus, window
# enumeration, metric layer and seeding all survived the move. If they differ, the
# corpus is not the same one -- almost certainly because fetch_nse.py was re-run.
#
# This asserts rather than printing an expected block for a human to eyeball: a
# check you have to read is a check that gets skipped when you are in a hurry.
import json  # noqa: E402

os.chdir(REPO)
before = set(glob.glob("results/*/results.json"))
r = subprocess.run(
    [
        sys.executable,
        "phase-a/eval/calibrate.py",
        "--split",
        "test",
        "--skip-model",
        "--no-figures",
    ],
    capture_output=True,
    text=True,
)
print(r.stdout[-1200:] or r.stderr[-1200:])
if r.returncode != 0:
    raise SystemExit(f"the baseline run failed (exit {r.returncode}); see the output above")

EXPECTED = {
    "last_value": {"crps": 89.0381, "cov80": 0.0008},
    "random_walk_drift": {"crps": 67.2363, "cov80": 0.8369},
}
EXPECTED_BLOCKS = 22

# Bind to the run this cell just produced, not to the newest on disk. Sorting by
# name picks whatever ran last, so a restored results/ or a re-run of this cell
# could pass the check against an *older* run while the new one silently failed.
new = set(glob.glob("results/*/results.json")) - before
if len(new) != 1:
    raise SystemExit(f"expected exactly one new run directory, got {sorted(new)}")
run_json = pathlib.Path(next(iter(new)))
print("checking", run_json.parent)
res = json.loads(run_json.read_text())

failures = []
for name, want in EXPECTED.items():
    got = res["models"][name]
    if abs(got["crps"] - want["crps"]) > 1e-3:
        failures.append(f"{name} CRPS {got['crps']:.4f} != {want['crps']}")
    cov = got["coverage"]["80"]["empirical"]
    if abs(cov - want["cov80"]) > 1e-4:
        failures.append(f"{name} cov@80 {cov:.4f} != {want['cov80']}")

blocks = res["models"]["last_value"]["effective_blocks"]
if blocks != EXPECTED_BLOCKS:
    failures.append(f"effective blocks {blocks} != {EXPECTED_BLOCKS}")

if failures:
    raise SystemExit("PORT CHECK FAILED:\n  " + "\n  ".join(failures))
print("\nPORT CHECK PASSED — corpus, windows, metrics and seeding all reproduce.")

In [ ]:
# ---- 5. run the grid -------------------------------------------------------
# Resumable: if a previous session left a run directory with a partial, point --resume
# at it and only the missing batches are recomputed. Resume is bit-identical, because
# each batch is seeded from its own offset.
RESUME_DIR = None  # e.g. "results/20260804T...."; None = fresh run

cmd = [
    sys.executable,
    "phase-a/eval/calibrate.py",
    "--split",
    SPLIT,
    "--batch-size",
    str(BATCH_SIZE),
    "--checkpoint-every",
    str(CHECKPOINT_EVERY),
]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if RESUME_DIR:
    cmd += ["--resume", RESUME_DIR]

print(" ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
print("exit code:", code)
if code:
    # Without this the analysis and packaging cells run on whatever happens to be in
    # results/, which under Save & Run All means a green notebook over a failed grid.
    raise SystemExit(code)

In [ ]:
# ---- 6. offline analysis (CPU only) ----------------------------------------
# subprocess rather than a `!` escape: shell escapes inside an if-block are fragile
# and do not survive tooling that parses the notebook as Python.
runs = sorted(glob.glob("results/*/ensembles.npz"))
if not runs:
    print("no ensembles.npz yet — run the grid cell first")
else:
    latest = str(pathlib.Path(runs[-1]).parent)
    print("analysing", latest, flush=True)
    r = subprocess.run(
        [sys.executable, "phase-a/eval/run_analysis.py", latest],
        capture_output=True,
        text=True,
    )
    print(r.stdout or r.stderr)

In [ ]:
# ---- 7. package results for download ---------------------------------------
# Only /kaggle/working survives the session. Zip results/ so it can be downloaded
# from the notebook's Output tab, then copied back into the local repo.
out = "/kaggle/working/kronos_results"
shutil.make_archive(out, "zip", root_dir=str(REPO), base_dir="results")
size = os.path.getsize(out + ".zip") / 2**20
print(f"{out}.zip  ({size:.1f} MB)")
print("\nDownload from the Output panel, unzip into the local repo, then:")
print("  uv run python phase-a/eval/run_analysis.py results/<run-dir>")

## If the session ends mid-run

Nothing is lost. Partials flush every `CHECKPOINT_EVERY` batches.

1. Download `kronos_results.zip` from the Output panel before the session expires.
2. In the next session, re-run cells 1–4, upload the zip as a dataset (or re-clone and
   restore `results/`), set `RESUME_DIR` to the run directory, and re-run cell 5.

Only the missing batches are recomputed, and the result is identical to an uninterrupted
run — `tests/test_resume.py` asserts that at `rtol=0, atol=0`.

## Tuning

| symptom | change |
|---|---|
| CUDA out of memory | `BATCH_SIZE` 24 → 12 → 6 |
| Want a quick smoke test first | `LIMIT = 60` (~5 min) |
| Session keeps dying | lower `CHECKPOINT_EVERY` to 2 |

Peak VRAM was 5.9 GB at batch 6 on a 6 GB card, so batch 24 on a 16 GB card should sit
comfortably inside budget.